In [1]:
import pandas as pd 
import os 
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd() , os.pardir)))

from utils.helpers import get_db_engine
from utils.loggers_config import logger

engine = get_db_engine()

query = "SELECT * FROM engineered_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

df.head()

[2026-05-12 10:11:29,461: INFO: 4007626463: Attempting to fetch data from PostgreSQL...]
[2026-05-12 10:11:32,617: INFO: 4007626463: Data ingestion successful! Loaded 440832 rows and 18 columns.]


,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Churn,Gender_male,Subscription Type_premium,Subscription Type_standard,is_critical_payment_delay,is_low_spender,high_support_risk,is_passive_user,is_low_usage,tenure_segment
0,47,38.0,12.0,10,8,0,250.0,22.0,1,False,False,True,0,1,1,1,0,4
1,39,2.0,27.0,10,24,0,348.0,1.0,1,False,False,False,1,1,1,0,0,1
2,60,21.0,19.0,7,11,0,718.0,16.0,1,True,True,False,0,0,1,1,0,3
3,22,35.0,25.0,8,22,0,805.0,8.0,1,False,True,False,1,0,1,0,0,4
4,19,14.0,16.0,6,14,0,776.0,9.0,1,False,False,True,0,0,1,0,0,3


In [2]:
query = "SELECT * FROM engineered_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

[2026-05-12 10:11:32,646: INFO: 1955847757: Attempting to fetch data from PostgreSQL...]
[2026-05-12 10:11:35,079: INFO: 1955847757: Data ingestion successful! Loaded 440832 rows and 18 columns.]


In [3]:
from sklearn.model_selection import train_test_split  
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
logger.info(f"Split complete. Training: {X_train.shape[0]} rows, Validation: {X_val.shape[0]} rows.")

[2026-05-12 10:11:37,074: INFO: 2529232317: Split complete. Training: 352665 rows, Validation: 88167 rows.]


In [4]:
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Final_Model_Comparison")

c:\Users\ASUS\customer-churn-mlops-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/05/12 10:11:40 INFO mlflow.tracking.fluent: Experiment with name 'Final_Model_Comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/ASUS/customer-churn-mlops-pipeline/notebooks/mlruns/2', creation_time=1778566300468, experiment_id='2', last_update_time=1778566300468, lifecycle_stage='active', name='Final_Model_Comparison', tags={}, workspace='default'>

In [5]:
from sklearn.linear_model import LogisticRegression
import yaml
    
with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    lr_params = config['logistic_regression']

with mlflow.start_run(run_name="Logistic_Regression"):
    lr_model = LogisticRegression(**lr_params)
    lr_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, lr_model.predict(X_train))
    val_acc = accuracy_score(y_val, lr_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.sklearn.log_model(lr_model, "model")
    logger.info(f"Logistic Regression - Train Accuracy: {train_acc:.4f}, Validation Accuracy: {val_acc:.4f}")

c:\Users\ASUS\customer-churn-mlops-pipeline\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/05/12 10:12:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 10:12:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended

[2026-05-12 10:12:07,657: INFO: 3526002368: Logistic Regression - Train Accuracy: 0.9674, Validation Accuracy: 0.9680]


In [7]:
from sklearn.ensemble import RandomForestClassifier

with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    rf_params = config['random_forest']

with mlflow.start_run(run_name="Random_Forest"):
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, rf_model.predict(X_train))
    val_acc = accuracy_score(y_val, rf_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.sklearn.log_model(rf_model, "model")
    logger.info(f"Random Forest - Train Accuracy: {train_acc:.4f}, Validation Accuracy: {val_acc:.4f}")

2026/05/12 10:14:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 10:14:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 10:14:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-05-12 10:14:55,130: INFO: 671431730: Random Forest - Train Accuracy: 0.9830, Validation Accuracy: 0.9833]


In [8]:
from xgboost import XGBClassifier

with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    xgb_params = config['xgboost']

with mlflow.start_run(run_name="XGBoost"):
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, xgb_model.predict(X_train))
    val_acc = accuracy_score(y_val, xgb_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.xgboost.log_model(xgb_model, "model")
    logger.info(f"XGBoost -> Train: {train_acc:.4f}, Val: {val_acc:.4f}")

c:\Users\ASUS\customer-churn-mlops-pipeline\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:15:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/12 10:15:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 10:15:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-05-12 10:15:31,970: INFO: 3181793713: XGBoost -> Train: 0.9875, Val: 0.9878]


In [9]:
from catboost import CatBoostClassifier

with open("../config/hyperparams.yaml", "r") as f: 
    config = yaml.safe_load(f)
    cb_params = config['catboost']

with mlflow.start_run(run_name="CatBoost"):
    cb_model = CatBoostClassifier(**cb_params)
    cb_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, cb_model.predict(X_train))
    val_acc = accuracy_score(y_val, cb_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.catboost.log_model(cb_model, "model")
    
    logger.info(f"CatBoost -> Train: {train_acc:.4f}, Val: {val_acc:.4f}")

2026/05/12 10:16:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 10:16:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-05-12 10:16:06,986: INFO: 3475300477: CatBoost -> Train: 0.9876, Val: 0.9879]


In [10]:
with mlflow.start_run(run_name="Final_Best_Model_CatBoost"):
    
    mlflow.catboost.log_model(cb_model, "model")
    mlflow.set_tag("model_type", "best_performer")
    mlflow.log_metrics({"final_accuracy": 0.9879}) 
    
    run_id = mlflow.active_run().info.run_id
    print(f"The best model has been registered. Run ID: {run_id}")

2026/05/12 11:48:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 11:48:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


The best model has been registered. Run ID: c3bf051446ac4753acf02ca370ce48a1


In [11]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, "catboost_churn_model.cbm")
cb_model.save_model(model_path)

print(f"The model is locally saved at: {model_path}")

The model is locally saved at: ../models\catboost_churn_model.cbm
